In [1]:
import socket
local_ip = socket.gethostbyname(socket.gethostname())
print(local_ip)  # 이 값으로 설정

192.168.34.1


In [2]:
import pyspark
print(pyspark.__version__)

4.0.2


In [3]:
import os
import sys
from dotenv import load_dotenv
from pyspark.sql import SparkSession

# .env 파일 로드하여 GCP 자격증명 정보 가져오기
load_dotenv()

# 환경 변수에서 PYSPARK_PYTHON 설정을 제거하여 도커(익스큐터)로 전달되는 것을 완전히 방지합니다.
os.environ.pop("PYSPARK_PYTHON", None)
os.environ.pop("PYSPARK_DRIVER_PYTHON", None)

gcp_key_path = "/llm/eth-proj-v2/gcp-key.json"
gcp_project_id = os.getenv("GCP_PROJECT_ID")

builder = (
    SparkSession.builder
    .appName("test_silver")
    .master("spark://localhost:7077")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.driver.host", "192.168.34.1")
    .config("spark.driver.memory", "3g")
    .config("spark.executor.memory", "4g")
    .config("spark.pyspark.python", "python3")
    .config("spark.pyspark.driver.python", sys.executable)
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.hadoop.fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
    .config("spark.hadoop.fs.AbstractFileSystem.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
    .config("spark.hadoop.google.cloud.auth.service.account.enable", "true")
    .config(
        "spark.jars.packages",
        "com.google.cloud.bigdataoss:gcs-connector:hadoop3-2.2.22,com.google.cloud.spark:spark-4.0-bigquery:0.44.1"
    )
)

builder = builder.config(
    "spark.hadoop.google.cloud.auth.service.account.json.keyfile",
    gcp_key_path
)

if gcp_project_id:
    builder = builder.config(
        "spark.hadoop.fs.gs.project.id",
        gcp_project_id
    )

spark = builder.getOrCreate()
print("Spark Session initialized successfully!")

desc = spark.sparkContext.setJobDescription


Spark Session initialized successfully!


In [4]:
from src.schema.bronze_schema import transaction_schema, receipt_schema
from pyspark.sql import SparkSession
from src.config import BUCKET_NAME, get_logger, GCS_BRONZE_PREFIX, GCS_SILVER_PREFIX


def read_bronze(spark: SparkSession, folder_name:str, dt:str, schema):
    spark.sparkContext.setLogLevel("ERROR")
    path = f"gs://{BUCKET_NAME}/{GCS_BRONZE_PREFIX}/{folder_name}/{dt}"
    return (spark.read
		.schema(schema)
		.option("mode", "FAILFAST")
		.json(path)
	)

txns = read_bronze(spark, "transactions", "dt=2026-05-01", schema=transaction_schema)
receipts = read_bronze(spark, "receipts", "dt=2026-05-01", schema=receipt_schema)

In [5]:
from pyspark.sql import SparkSession
from src.silver.spark_config import WEI_PER_ETH, read_bronze, get_logger, get_spark_session
from src.silver.utils import write_silver
from src.schema.bronze_schema import transaction_schema, receipt_schema
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType





receipts_slim = receipts.select(
	F.col("transaction_hash"),
	F.col("status")
)

enriched = (
	txns
	.join(receipts_slim, txns["hash"] == receipts_slim["transaction_hash"], how="left")
	.drop("transaction_hash")
)

enriched = (
	enriched.withColumn(
		"value_eth",
		(F.col("value").cast(DecimalType(38, 0)) / F.lit(WEI_PER_ETH)).cast(DecimalType(38, 18))
	).withColumn(
		"is_success",
		F.col("status") == 1
	).withColumn(
		"dt",
		F.to_date(F.from_unixtime(F.col("block_timestamp")))
	)
)

final_cols = [
		"hash", "block_number", "block_timestamp",
		"from_address", "to_address",
		"value_eth", "is_success", "dt"
]


df = enriched.select(*final_cols)

In [6]:
desc("enriched txs")
df.write.format("noop").mode("overwrite").save()
desc("")

## 고래지갑

In [7]:
import os
import src.schema.bronze_schema as bronze_schema
from pyspark.sql import SparkSession
from src.config import BUCKET_NAME, get_logger, GCS_BRONZE_PREFIX, GCS_SILVER_PREFIX

def read_silver(spark: SparkSession, folder_name:str, dt: str, schema):
    spark.sparkContext.setLogLevel("ERROR")
    path = f"gs://{BUCKET_NAME}/{GCS_SILVER_PREFIX}/{folder_name}/{dt}"
    return (spark.read
		.schema(schema)
		.option("mode", "FAILFAST")
		.parquet(path)
	)

In [8]:
from src.silver.known_labels import load_address_labels
from pyspark.sql import functions as F
from pyspark.sql import Window, SparkSession, DataFrame

"""
고래 트랜잭션 뷰 생성
----------------
1. txn_enriched에서 threshold 이상 필터
2. from/to 각각 라벨 조인
3. 주소별 당일 누적 통계 (window)
4. 이상 패턴 플래그
"""
from src.schema.silver_schema import enriched_transaction_schema

txns = read_silver(spark, "txn_enriched", "dt=2026-05-01", schema=enriched_transaction_schema)



whales = (
	txns
	.filter(F.col("value_eth") >= 100)
	.filter(F.col("is_success") == True)
)

labels = load_address_labels(spark)
labels_bc = F.broadcast(labels)

whales = (
	whales.join(
		labels_bc.select(
			F.col("address").alias("from_address"),
			F.col("label_name").alias("from_label"),
			F.col("label_category").alias("from_category")
		),
		on="from_address", how="left"
	).join(
		labels_bc.select(
			F.col("address").alias("to_address"),
			F.col("label_name").alias("to_label"),
			F.col("label_category").alias("to_category"),
		),
		on="to_address", how="left"
	)
)

# 라벨 없으면 "Unknown" 처리 및 엔티티(Entity) 추출
whales = (
	whales
	.withColumn("from_label",    F.coalesce("from_label",    F.lit("Unknown")))
	.withColumn("from_category", F.coalesce("from_category", F.lit("Unknown")))
	.withColumn("to_label",      F.coalesce("to_label",      F.lit("Unknown")))
	.withColumn("to_category",   F.coalesce("to_category",   F.lit("Unknown")))
	# 핵심 브랜드명만 추출 (예: Binance 14 -> Binance, Uniswap V3: Router -> Uniswap V3)
	.withColumn("from_entity", F.trim(F.regexp_extract(F.col("from_label"), r"^([^0-9:#]+)", 1)))
	.withColumn("to_entity",   F.trim(F.regexp_extract(F.col("to_label"),   r"^([^0-9:#]+)", 1)))
)

# --- 1. 기초 인텔리전스 추출 ---
whales = (
	whales
	.withColumn("hour", F.hour(F.from_unixtime("block_timestamp").cast("timestamp")))
	# 보낸 쪽/받는 쪽이 모두 Unknown이면 순수 개인 고래 거래로 간주
	.withColumn("is_private_transaction", 
		(F.col("from_category") == "Unknown") & (F.col("to_category") == "Unknown"))
)

# --- 2. 고래 체급 분류 (Whale Tier) ---
whales = whales.withColumn("whale_tier", 
	F.when(F.col("value_eth") >= 10000, "Humpback")
	.when(F.col("value_eth") >= 1000,  "Whale")
	.when(F.col("value_eth") >= 100,   "Shark")
	.otherwise("Crab")
)

# --- 3. 자금 흐름 성격 규정 (Flow Type) ---
whales = whales.withColumn("flow_type",
	F.when((F.col("from_category") == "Unknown") & (F.col("to_category") == "CEX"), "CEX_DEPOSIT")    # 매도 압력
	.when((F.col("from_category") == "CEX") & (F.col("to_category") == "Unknown"), "CEX_WITHDRAWAL") # 매수 압력
	.when((F.col("from_category") == "DEX") | (F.col("to_category") == "DEX"),      "DEX_TRADE")      # 스왑 활동
	.when((F.col("from_category") == "Bridge") | (F.col("to_category") == "Bridge"), "BRIDGE_MOVE")   # 자산 이동
	.when((F.col("from_category") == "Unknown") & (F.col("to_category") == "Unknown"), "PRIVATE_MOVE") # 고래간 이동
	.otherwise("OTHER")
)

# --- 4. 누적 통계 및 플래그 (중간 연산용) ---
w_from = Window.partitionBy("from_address", "dt").orderBy("block_timestamp") \
	.rowsBetween(Window.unboundedPreceding, Window.currentRow)

w_to = Window.partitionBy("to_address", "dt").orderBy("block_timestamp") \
	.rowsBetween(Window.unboundedPreceding, Window.currentRow)

whales = whales \
	.withColumn("cumul_sent_eth", F.sum("value_eth").over(w_from)) \
	.withColumn("cumul_tx_count",  F.count("hash").over(w_from)) \
	.withColumn("cumul_recv_eth", F.sum("value_eth").over(w_to))

# 고빈도 송금 플래그
whales = whales.withColumn("flag_high_freq", 
	(F.col("cumul_tx_count") >= 5) & (F.col("cumul_sent_eth") >= 500))

# --- 5. 최종 컬럼 선택 (골드 레이어 준비물) ---
final_cols = [
	"hash", "block_timestamp", "hour", "dt",
	"from_address", "from_label", "from_entity", "from_category",
	"to_address",   "to_label",   "to_entity",   "to_category",
	"value_eth",
	"cumul_sent_eth", "cumul_tx_count", "cumul_recv_eth",
	"whale_tier", "flow_type", "is_private_transaction",
	"flag_high_freq"
]

df = whales.select(*final_cols)

In [9]:
df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [hash#38, block_timestamp#40L, hour#65, dt#45, from_address#41, from_label#59, from_entity#63, from_category#60, to_address#42, to_label#61, to_entity#64, to_category#62, value_eth#43, cumul_sent_eth#69, cumul_tx_count#71L, cumul_recv_eth#73, whale_tier#67, flow_type#68, is_private_transaction#66, ((cumul_tx_count#71L >= 5) AND (cumul_sent_eth#69 >= 500.000000000000000000)) AS flag_high_freq#75]
   +- Window [sum(value_eth#43) windowspecdefinition(to_address#42, dt#45, block_timestamp#40L ASC NULLS FIRST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS cumul_recv_eth#73], [to_address#42, dt#45], [block_timestamp#40L ASC NULLS FIRST]
      +- Sort [to_address#42 ASC NULLS FIRST, dt#45 ASC NULLS FIRST, block_timestamp#40L ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(to_address#42, dt#45, 16), ENSURE_REQUIREMENTS, [plan_id=175]
            +- Window [sum(value_eth#43) windowspec

In [10]:
desc("whale silver")
df.write.format("noop").mode("overwrite").save()
desc("")

### token flow

In [11]:
from pyspark.sql import SparkSession
from pathlib import Path

# [수정] 드라이브 문자(C:)를 제외하여 Windows와 Docker 컨테이너 둘 다 인식 가능한 절대 경로로 설정
DATA_DIR = Path("/llm/eth-proj-v2/src/data")
KNOWN_LABELS_PATH = DATA_DIR / "known_labels.parquet"
TOKEN_META_PATH   = DATA_DIR / "top1000_erc20_tokens.parquet"

# ERC-20 전송 이벤트 시그니처 (Transfer topic0)
TRANSFER_TOPIC = "0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4a11628f55a4df523b3ef"

# 파일에 없지만 분석에 꼭 필요한 특수 주소들 (수동 관리)
MANUAL_LABELS = {
    "0x000000000000000000000000000000000000dead": ("Dead Address", "Burn"),
    "0x0000000000000000000000000000000000000000": ("Null Address", "Burn"),
}

def load_token_metadata(spark: SparkSession):
    """
    수집된 토큰 메타데이터(심볼, 데시멀 등) 데이터를 Spark DataFrame으로 로드
    컬럼: token_address, symbol, token_name, decimals
    """
    if not TOKEN_META_PATH.exists():
        print(f"⚠ Warning: {TOKEN_META_PATH} 파일이 없습니다. 빈 DF를 반환합니다.")
        return spark.createDataFrame([], "token_address string, symbol string, token_name string, decimals int")

    # [수정] .as_posix()를 사용하여 역슬래시(\)가 아닌 리눅스 형태의 슬래시(/) 경로로 Spark에 전달합니다.
    return spark.read.parquet(TOKEN_META_PATH.as_posix()) \
        .selectExpr("address as token_address", "symbol", "name as token_name", "decimals")


In [12]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from src.silver.spark_config import get_spark_session, get_logger
from src.silver.utils import write_silver
from src.schema.bronze_schema import token_transfer_schema, block_schema, receipt_schema
from src.silver.known_labels import load_address_labels




# 1. 소스 데이터 로드 (contracts는 제외)
transfers = read_bronze(spark, "token_transfers", "dt=2026-05-01", token_transfer_schema)
blocks = read_bronze(spark, "blocks", "dt=2026-05-01", block_schema)
receipts = read_bronze(spark, "receipts", "dt=2026-05-01", receipt_schema)

# 2. 마스터 데이터(Dimension) 로드
token_meta = load_token_metadata(spark)
address_labels = load_address_labels(spark)

# 데이터 가공 및 조인
blocks_slim = blocks.select(
    F.col("number").alias("block_number"),
    F.col("timestamp").alias("block_timestamp")
)

receipts_slim = receipts.select(
    F.col("transaction_hash"),
    F.col("status")
)

# 토큰 정보 조인 (Inner Join: 우리가 관리하는 주요 토큰들만 분석 대상)
flow = transfers.join(
    F.broadcast(token_meta),
    on="token_address",
    how="inner"
)

# 블록 시간 조인
flow = flow.join(F.broadcast(blocks_slim), on="block_number", how="left")

# [수정] 영수증(Receipts) 조인 시 F.broadcast 제거 (대용량 테이블이므로 셔플 조인이 맞음)
flow = flow.join(
    receipts_slim,
    on="transaction_hash",
    how="inner"
).filter(F.col("status") == 1)

# 기초 인텔리전스 (시간, 수량 계산)
flow = (
    flow
    .withColumn("hour", F.hour(F.from_unixtime("block_timestamp").cast("timestamp")))
    .withColumn("dt", F.to_date(F.from_unixtime("block_timestamp")))
    .withColumn(
        "amount",
        (F.col("value") / F.pow(F.lit(10.0), F.col("decimals").cast(DoubleType()))).cast(DoubleType())
    )
)

# 4. 주소 라벨링 (보낸 쪽/받는 쪽)
# [수정] 크기가 매우 작은 라벨 데이터(약 440KB)를 브로드캐스트하여 무거운 셔플/정렬 조인 제거
labels_bc = F.broadcast(address_labels)

# 발신자 라벨
flow = flow.join(
    labels_bc.selectExpr("address as from_address", "label_name as from_label", "label_category as from_category"),
    on="from_address", how="left"
).fillna({"from_label": "Unknown", "from_category": "Unknown"})

# 수신자 라벨
flow = flow.join(
    labels_bc.selectExpr("address as to_address", "label_name as to_label", "label_category as to_category"),
    on="to_address", how="left"
).fillna({"to_label": "Unknown", "to_category": "Unknown"})

# 5. 최종 컬럼 선택
final_cols = [
    "transaction_hash", "status", "block_timestamp", "hour", "dt",
    "token_address", "symbol", "token_name",
    "from_address", "from_label", "from_category",
    "to_address", "to_label", "to_category",
    "amount"
]

df = flow.select(*final_cols)


2026-05-30 15:21:02,258 [INFO] [ETL_App.Read Bronze Layer token_transfers / dt=2026-05-01] [Read_Bronze] Reading from gs://chan-ethe-data/bronze/token_transfers/dt=2026-05-01
2026-05-30 15:21:02,611 [INFO] [ETL_App.Read Bronze Layer blocks / dt=2026-05-01] [Read_Bronze] Reading from gs://chan-ethe-data/bronze/blocks/dt=2026-05-01
2026-05-30 15:21:02,947 [INFO] [ETL_App.Read Bronze Layer receipts / dt=2026-05-01] [Read_Bronze] Reading from gs://chan-ethe-data/bronze/receipts/dt=2026-05-01


In [13]:
desc("token flow silver")
df.write.format("noop").mode("overwrite").save()
desc("")

In [15]:
spark.stop()